# 00 · Téléchargement et nettoyage du jeu de données

Pipeline seulement, pas d'analyse (voir `01_eda.ipynb`). Pour chaque paire
`(competition_id, season_id)` de `PAIRS`, ce notebook :

1. télécharge les événements StatsBomb de tous les matchs (cache JSON dans `data/raw/statsbomb/`,
   chaque fichier n'est téléchargé qu'une fois) ;
2. les aplatit en table d'actions et applique le nettoyage (`xt/data/clean.py`) ;
3. écrit `data/processed/actions_<competition_id>_<season_id>.parquet`.

Si le parquet existe déjà, il est simplement relu. Pour reconstruire après un changement dans
`clean.py`, mettre `REBUILD = True`.

In [1]:
from xt.data import build_actions, load_competitions
from xt.data.clean import PROCESSED_DIR

In [2]:
# (competition_id, season_id)
# Coupe du monde 2022 : (43, 106)        Premier League 2015/16 : (2, 27)
# 4 championnats complets 2015/16 : (2, 27), (11, 27), (12, 27), (7, 27)  (~1 500 matchs, ~4 Go de JSON)
# (Bundesliga 2015/16 (9, 27) ne contient que les 34 matchs de Leverkusen : à éviter)
PAIRS = [(2, 27), (11, 27), (12, 27), (7, 27), (43, 106), (55, 282)]

REBUILD = True   # True : réécrit les parquets (après une modification du nettoyage)
N_WORKERS = 8     # téléchargements en parallèle

In [3]:
# Optionnel : toutes les compétitions-saisons disponibles, pour choisir PAIRS
# load_competitions()[["competition_id", "season_id", "competition_name", "season_name"]]

In [4]:
if REBUILD:
    for c, s in PAIRS:
        (PROCESSED_DIR / f"actions_{c}_{s}.parquet").unlink(missing_ok=True)

for c, s in PAIRS:
    df = build_actions(c, s, n_workers=N_WORKERS)
    print(f"({c}, {s}) {df.competition_name.iloc[0]} {df.season_name.iloc[0]} : "
          f"{df.match_id.nunique()} matchs, {len(df):,} actions")

events:   0%|          | 0/380 [00:00<?, ?it/s]

(2, 27) Premier League 2015/2016 : 380 matchs, 771,123 actions


events:   0%|          | 0/380 [00:00<?, ?it/s]

(11, 27) La Liga 2015/2016 : 380 matchs, 763,657 actions


events:   0%|          | 0/380 [00:00<?, ?it/s]

(12, 27) Serie A 2015/2016 : 380 matchs, 775,561 actions


... (2 outputs left out)

events:   0%|          | 0/64 [00:00<?, ?it/s]

(43, 106) FIFA World Cup 2022 : 64 matchs, 140,280 actions


events:   0%|          | 0/51 [00:00<?, ?it/s]

(55, 282) UEFA Euro 2024 : 51 matchs, 110,807 actions


In [5]:
for c, s in PAIRS:
    p = PROCESSED_DIR / f"actions_{c}_{s}.parquet"
    print(f"{p.relative_to(PROCESSED_DIR.parents[1])}  {p.stat().st_size / 1e6:.1f} Mo")

data\processed\actions_2_27.parquet  37.8 Mo
data\processed\actions_11_27.parquet  37.4 Mo
data\processed\actions_12_27.parquet  38.1 Mo
data\processed\actions_7_27.parquet  38.2 Mo
data\processed\actions_43_106.parquet  6.9 Mo
data\processed\actions_55_282.parquet  5.5 Mo
